[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AndresOrdonez369/deepLearningWeek1/blob/main/proyectos/NB02_sabado_tu_primera_red.ipynb)

# NB02 · Tu primera red: de la cuenta a mano a Keras
**Neural Networks: Deep Learning · UIDE · Semana 1 · Sábado** — Docente: Andrés Ordóñez-Bolaños

Hoy vas a ver **cómo aprende una red completa**. Primero hacemos a mano las cuentas de una red pequeñita (las mismas de las láminas), después conocemos **TensorFlow** y **Keras** desde cero y terminamos entrenando redes de verdad en pocas líneas, con gráficas de todo.

### Cómo usar este Colab (2 minutos)
1. **Archivo → Guardar una copia en Drive.** Así tus cambios quedan guardados.
2. Ejecuta la celda **Configuración** (tarda unos segundos: carga TensorFlow). Después avanza en orden con `Shift + Enter`.
3. Si algo raro pasa: **Entorno de ejecución → Ejecutar todo**.
4. Cada parte sigue el mismo orden: **Qué vas a hacer → la idea → ejemplo resuelto → Qué deberías ver → Tu turno**.
5. En **Tu turno** reemplaza los `None` y ejecuta la celda de revisión: ✅ correcto · ❌ todavía no (con una pista) · ⏳ todavía no lo completas.
6. **No tiene nota.** Trabajen en parejas y equivóquense sin miedo.

**La red de las cuentas a mano:** 2 entradas → 2 neuronas ocultas → 1 salida, todas con sigmoide.

### Mapa del Colab
| Parte | Qué aprendes | Tu turno | Lámina |
|---|---|---|---|
| 0 | NumPy exprés: vectores, matrices y una capa en una línea (si ya lo dominas, sáltala) | 0 | 42 |
| T | Conoce tus herramientas: qué son TensorFlow y Keras | — | 53, 61, 62 |
| A | Forward: la red predice | TA | 48 |
| B | Backward: repartir la culpa | TB | 49–51 |
| C | Un paso de aprendizaje | TC | 52 |
| D | TensorFlow calcula lo mismo | TD | 53 |
| E | Keras paso a paso: una capa oculta resuelve XOR | TE | 43, 46, 63 |
| F | Keras completo con las lunas · **Mini-reto 2** (los círculos) | 🏁 | 64–67 |
| G | ¿Qué optimizador llega primero? | TG | 58 |
| 🏁 | Resumen y **Tareas finales** | 2 tareas | 68 |

En clase (lámina 68): prioridad **A, B, E y F**. C, D y G pueden quedar para casa.

In [ ]:
# @title Configuración y autocorrección · ejecuta esta celda primero (no necesitas abrirla)
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")   # hide TensorFlow info messages
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import tensorflow as tf
import keras
from keras import layers
from sklearn.datasets import make_moons, make_circles
from sklearn.model_selection import train_test_split

SEED = 2026
keras.utils.set_random_seed(SEED)
np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3, "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})

# The same colors as the slides: amber = yes / class 1, azure = no / class 0, mint = learned, coral = error
AMBAR, AZUL, MENTA, CORAL, TINTA, LILA = "#E39A12", "#2B8FD6", "#12A37F", "#E0533D", "#1F2A3A", "#8B6FD1"
MAPA = LinearSegmentedColormap.from_list("si_no", [AZUL, "#F4F7FA", AMBAR])

# ---------- self-check helpers (formative feedback, no grades) ----------
def _ok(msg):   print("✅ " + msg)
def _no(msg):   print("❌ " + msg)
def _todo():    print("⏳ Aún no completas esta parte: reemplaza los None y vuelve a ejecutar la celda.")
def _f(v):
    try:
        return np.asarray(v, dtype=float)
    except Exception:
        return None
def _near(a, b, tol=1e-3):
    try:
        return bool(np.allclose(np.asarray(a, dtype=float).reshape(-1), np.asarray(b, dtype=float).reshape(-1), atol=tol))
    except Exception:
        return False
_s = lambda z: 1 / (1 + np.exp(-z))
_W1 = np.array([[0.5, -0.3], [0.2, 0.8]]); _b1 = np.array([0.1, -0.1])
_W2 = np.array([[0.7, -0.4]]);             _b2 = np.array([0.05])
_x2 = np.array([0.2, 1.0]); _y2 = 0.0
_a1 = _s(_W1 @ _x2 + _b1); _yh = _s(_W2 @ _a1 + _b2)
_d2 = _yh - _y2; _dW2 = np.outer(_d2, _a1); _d1 = (_W2.T @ _d2) * _a1 * (1 - _a1); _dW1 = np.outer(_d1, _x2)
_yh_new = _s((_W2 - .5 * _dW2) @ _s((_W1 - .5 * _dW1) @ _x2 + (_b1 - .5 * _d1)) + (_b2 - .5 * _d2))

def revisar_0(z_capa, a_capa):
    if z_capa is None or a_capa is None:
        return _todo()
    if not _near(z_capa, [0.45, 0.50]):
        return _no(f"z_capa debería ser W @ x + b = [0.45 0.5] y es {_f(z_capa)}. ¿Sumaste b?")
    if not _near(a_capa, _s(np.array([0.45, 0.5]))):
        return _no("a_capa es la sigmoide de z_capa: sigmoid(z_capa).")
    _ok("[0.6106 0.6225]: dos neuronas calculadas en una línea. Guárdalo: es la capa oculta de la Parte A.")

def revisar_ta(a1_2, y_hat_2, loss_2):
    if a1_2 is None or y_hat_2 is None or loss_2 is None:
        return _todo()
    if not _near(a1_2, _a1):
        return _no(f"La capa oculta debería dar a ≈ {np.round(_a1, 4)}. Recuerda: a = sigmoid(W1 @ x + b1).")
    if not _near(y_hat_2, _yh):
        return _no(f"ŷ debería ser ≈ {_yh[0]:.4f}. Usa a1_2 como entrada de la salida: sigmoid(W2 @ a1_2 + b2).")
    if _near(loss_2, -np.log(_yh)):
        return _no("Ojo: este socio NO pagó (y = 0). La pérdida es -log(1 - ŷ), no -log(ŷ).")
    if not _near(loss_2, -np.log(1 - _yh)):
        return _no(f"La pérdida debería ser -log(1 - ŷ) ≈ {float(-np.log(1 - _yh[0])):.4f}.")
    _ok(f"¡Bien! La red le da {_yh[0]:.0%} de probabilidad de pago a alguien que no pagó: pérdida {float(-np.log(1 - _yh[0])):.3f}.")

def revisar_tb(d2_2, dW2_2):
    if d2_2 is None or dW2_2 is None:
        return _todo()
    if not _near(d2_2, _d2):
        return _no(f"La culpa de la salida es ŷ - y = {_yh[0]:.4f} - 0 = {_d2[0]:.4f}.")
    if not _near(dW2_2, _dW2):
        return _no(f"El gradiente de W2 es culpa × lo que entró: {_d2[0]:.4f} × a1 ≈ {np.round(_dW2.ravel(), 4)}.")
    _ok("Correcto. La culpa es positiva: la red debe BAJAR su predicción para este socio.")

def revisar_tc(y_hat_2_nuevo):
    if y_hat_2_nuevo is None:
        return _todo()
    if not _near(y_hat_2_nuevo, _yh_new):
        return _no(f"Después del paso, ŷ debería ser ≈ {_yh_new[0]:.4f}. ¿Actualizaste las 4 piezas (W1, b1, W2, b2) con η = 0.5?")
    _ok(f"¡La red aprendió! ŷ bajó de {_yh[0]:.4f} a {_yh_new[0]:.4f}: ahora duda más de que este socio pague, como debe ser.")

def revisar_td(grad_engranajes):
    if grad_engranajes is None:
        return _todo()
    if not _near(grad_engranajes, 18.0):
        return _no(f"La derivada de (3x)² en x = 1 es 18 (×3 por dentro, ×6 por fuera). Obtuviste {float(_f(grad_engranajes))}.")
    _ok("18 = 3 × 6: TensorFlow aplicó la regla de la cadena, como los engranajes.")

def revisar_te(contar_parametros):
    if contar_parametros is None:
        return _todo()
    try:
        r = [contar_parametros(c) for c in ([2, 16, 16, 1], [10, 8, 8, 1], [3, 4, 1])]
    except Exception as e:
        return _no(f"Tu función produjo un error: {e}")
    if r[0] is None:
        return _todo()
    if r != [337, 169, 21]:
        return _no(f"Esperaba 337, 169 y 21; obtuve {r}. Pista: en cada capa, entradas × neuronas + neuronas.")
    _ok("337 (red de las lunas), 169 (10 → 8 → 8 → 1) y 21 (3 → 4 → 1). Igual que model.summary().")

def revisar_mini_reto_2(modelo, X_va, y_va):
    if modelo is None:
        return _todo()
    try:
        acc = float(np.mean((modelo.predict(X_va, verbose=0).ravel() >= 0.5) == y_va))
    except Exception as e:
        return _no(f"No pude evaluar tu modelo: {e}")
    print(f"Exactitud en validación: {acc:.1%}")
    if acc >= 0.90:
        _ok("¡Superaste el 90 %! Tu red aprendió una frontera circular.")
    elif acc < 0.6:
        _no("Cerca del 50 % = adivinar. ¿Tu red tiene capas ocultas con activación (relu o tanh)?")
    else:
        _no("Vas bien. Prueba más neuronas, más épocas o learning_rate=0.01.")

def revisar_tg(peor_optimizador, resultados):
    if peor_optimizador is None:
        return _todo()
    worst = max(resultados, key=resultados.get)
    if str(peor_optimizador).strip().lower() != worst.lower():
        return _no("Mira el final de las curvas (o el diccionario resultados_g): ¿cuál quedó más arriba?")
    _ok(f"Correcto: {worst} terminó con la pérdida más alta. Sin impulso ni pasos adaptativos, avanza lento.")

NUEVOS = np.array([[0.0, 1.0], [1.0, -0.5], [0.5, 0.25]])     # three new points for the final task
def revisar_tf1(modelo, probs_nuevos, clases_nuevas):
    if probs_nuevos is None or clases_nuevas is None:
        return _todo()
    p = modelo.predict(NUEVOS, verbose=0).ravel()
    if not _near(np.ravel(probs_nuevos), p, 1e-4):
        return _no("probs_nuevos debería salir de model.predict(NUEVOS). ¿Usaste el modelo de la Parte F?")
    if not np.array_equal(np.ravel(clases_nuevas).astype(int), (p >= 0.5).astype(int)):
        return _no("clases_nuevas: 1 si la probabilidad es ≥ 0.5, si no 0. Pista: (probs_nuevos >= 0.5).astype(int)")
    _ok(f"Probabilidades {np.round(p, 2)} → clases {(p >= 0.5).astype(int)}. Así se usa un modelo ya entrenado.")

def plot_boundary(model, X, y, ax, title):
    # Shaded = probability of class 1 (amber) or class 0 (azure); black line = 50 %
    g1, g2 = np.meshgrid(np.linspace(X[:, 0].min() - .5, X[:, 0].max() + .5, 200),
                         np.linspace(X[:, 1].min() - .5, X[:, 1].max() + .5, 200))
    P = model.predict(np.c_[g1.ravel(), g2.ravel()], verbose=0).reshape(g1.shape)
    ax.contourf(g1, g2, P, levels=np.linspace(0, 1, 11), cmap=MAPA, alpha=.75)
    ax.contour(g1, g2, P, levels=[.5], colors=TINTA, linewidths=2.5)
    ax.scatter(X[:, 0], X[:, 1], c=np.where(np.asarray(y) == 1, AMBAR, AZUL), edgecolor=TINTA, s=30, lw=.5)
    ax.set_title(title)

def dibujar_red(W1, b1, W2, b2, x=None, a1=None, y_hat=None, title="", color_w=None):
    # A 2-2-1 network: edge color = sign (amber +, azure -), edge width = size of the weight
    P = {"x1": (0, .8), "x2": (0, .2), "h1": (1, .8), "h2": (1, .2), "o": (2, .5)}
    fig, ax = plt.subplots(figsize=(8, 4.2))
    edges = [("x1", "h1", W1[0, 0]), ("x2", "h1", W1[0, 1]), ("x1", "h2", W1[1, 0]), ("x2", "h2", W1[1, 1]),
             ("h1", "o", W2[0, 0]), ("h2", "o", W2[0, 1])]
    for a, b_, wv in edges:
        (xa, ya), (xb, yb) = P[a], P[b_]
        ax.plot([xa, xb], [ya, yb], color=color_w or (AMBAR if wv > 0 else AZUL), lw=1 + 4 * abs(wv), alpha=.85, zorder=1)
        t = .3; ax.text(xa + (xb - xa) * t, ya + (yb - ya) * t + .045, f"{wv:+.2f}", ha="center", fontsize=10,
                        color=TINTA, bbox=dict(boxstyle="round,pad=.15", fc="white", ec="none"))
    vals = {"x1": None if x is None else x[0], "x2": None if x is None else x[1],
            "h1": None if a1 is None else a1[0], "h2": None if a1 is None else a1[1], "o": None if y_hat is None else float(np.ravel(y_hat)[0])}
    bias = {"h1": b1[0], "h2": b1[1], "o": float(np.ravel(b2)[0])}
    for k, (xp, yp) in P.items():
        ax.scatter(xp, yp, s=1700, color="white", edgecolor=TINTA, lw=2, zorder=2)
        ax.text(xp, yp, "" if vals[k] is None else f"{vals[k]:.2f}", ha="center", va="center", fontsize=12, weight="bold", zorder=3)
        if k in bias:
            ax.text(xp, yp + .15, f"b = {bias[k]:+.2f}", ha="center", fontsize=9, color="gray")
    for k, lab in [("x1", "antigüedad"), ("x2", "carga de la cuota"), ("h1", "oculta 1"), ("h2", "oculta 2"), ("o", "salida ŷ")]:
        ax.text(P[k][0], P[k][1] - .17, lab, ha="center", fontsize=9, color="gray")
    ax.set_xlim(-.4, 2.4); ax.set_ylim(-.05, 1.08); ax.axis("off"); ax.set_title(title)
    plt.show()

print("Listo ✔  TensorFlow", tf.__version__, "| Keras", keras.__version__, "· autocorrección cargada")

---
## 0 · NumPy exprés para redes (si ya lo dominas, sáltala)
📽 Lámina 42

**Qué vas a hacer:** escribir **una capa entera de neuronas en una sola línea** de NumPy.

**La idea:** el viernes una neurona hacía $z = \mathbf{w}\cdot\mathbf{x} + b$. Una **capa** son varias neuronas que miran los mismos datos. Si ponemos los pesos de cada neurona en una **fila** de una matriz $W$, toda la capa es:

$$\mathbf{z} = W\mathbf{x} + \mathbf{b} \qquad\qquad \mathbf{a} = \sigma(\mathbf{z})$$

**Léelo así:** "cada fila de $W$ (una neurona) multiplica a $\mathbf{x}$ y suma; a cada resultado le sumamos su sesgo; y la sigmoide se aplica a cada número por separado". En NumPy: `W @ x + b`.

In [ ]:
x = np.array([1.0, 0.5])               # one member: 2 features  -> shape (2,)
W = np.array([[0.5, -0.3],             # row 0: weights of hidden neuron 1
              [0.2,  0.8]])            # row 1: weights of hidden neuron 2  -> shape (2, 2)
b = np.array([0.1, -0.1])              # one bias per neuron

print("forma de x:", x.shape, "| forma de W:", W.shape, "| forma de b:", b.shape)
print("Fila 0 de W (neurona 1):", W[0], "| fila 1 (neurona 2):", W[1])
print("W @ x =", W @ x, " ← cada neurona hace su suma ponderada")

**Qué deberías ver:** formas `(2,)`, `(2, 2)` y `(2,)`, y `W @ x = [0.35 0.6]`: la neurona 1 da 0.5·1.0 − 0.3·0.5 = 0.35 y la neurona 2, 0.2·1.0 + 0.8·0.5 = 0.6.

### Tu turno 0 · Una capa en una línea
1. Suma los sesgos: `z_capa = W @ x + b`.
2. Aplica la sigmoide a cada número: `a_capa = sigmoid(z_capa)` (la función ya está en la celda).

In [ ]:
sigmoid = lambda z: 1 / (1 + np.exp(-z))    # works on whole arrays, number by number
z_capa = None    # TODO
a_capa = None    # TODO
print("z =", z_capa, "| a =", a_capa)

In [ ]:
revisar_0(z_capa, a_capa)

---
## T · Conoce tus herramientas: TensorFlow y Keras
📽 Láminas 61–62

**Qué vas a hacer:** entender qué son las dos librerías que usa todo el curso, cómo se importan y en qué se diferencian de NumPy.

**TensorFlow** es la librería de Google para hacer las cuentas de las redes neuronales (la publicó como código abierto en noviembre de 2015). Sabe hacer dos cosas que NumPy no hace solo: **calcular derivadas automáticamente** y repartir el trabajo en **CPU o GPU**.

**Keras** es la librería que te deja **armar y entrenar redes en pocas líneas** (la creó François Chollet en marzo de 2015; hoy es Keras 3). Por dentro usa a TensorFlow.

> **Analogía:** TensorFlow es **el motor**; Keras es **el volante y los pedales**. Puedes manejar sin abrir el capó, pero ayuda saber que el motor existe.

| Capa | Qué hace | En este curso |
|---|---|---|
| Tú (Python) | escribes qué quieres | celdas de Colab |
| **Keras** | piezas listas: capas, pérdidas, optimizadores, `fit` | Partes E, F, G |
| **TensorFlow** | tensores, derivadas automáticas, CPU/GPU | Parte D |
| Hardware | hace las multiplicaciones | la CPU o GPU de Colab |

En Colab ya vienen instaladas: solo hay que **importarlas**.

In [ ]:
import tensorflow as tf          # "tf" is the standard nickname
import keras                     # Keras 3
from keras import layers         # the building blocks (Dense, ...) -> we will write layers.Dense

print("TensorFlow", tf.__version__, "| Keras", keras.__version__)
print("GPU disponible:", tf.config.list_physical_devices("GPU") or "no (CPU basta para esta clase)")

### Tensores: los arrays de TensorFlow
Un **tensor** es como un array de NumPy (un bloque de números con forma), pero vive en TensorFlow y puede ir a la GPU. Una `tf.Variable` es un tensor que **puede cambiar**: así se guardan los pesos que la red aprende.

In [ ]:
t = tf.constant([[1.0, 2.0],
                 [3.0, 4.0]])               # a 2x2 tensor
print("tensor:\n", t)
print("forma:", t.shape, "| tipo:", t.dtype)
print("t @ t =\n", (t @ t).numpy())          # same matrix product as NumPy; .numpy() converts back

peso = tf.Variable(0.7)                      # a trainable number (a weight)
peso.assign(0.84)                            # training changes it like this
print("peso después de un paso:", peso.numpy())

**Qué deberías ver:** la forma `(2, 2)`, el producto `[[7. 10.] [15. 22.]]` (igual que en NumPy) y el peso que pasa de 0.7 a 0.84, como en la lámina 52.

<details><summary>Para curiosos: Keras 3 tiene varios motores</summary>

Keras 3 puede usar como motor a TensorFlow, a JAX o a PyTorch. En Colab, por defecto, usa TensorFlow. El código de Keras que escribas hoy funciona igual con cualquiera.
</details>

---
## Parte A · Forward: la información viaja hacia adelante
📽 Lámina 48

**Qué vas a hacer:** calcular a mano la predicción de una red 2 → 2 → 1.

**La idea:** cada capa hace lo de la Parte 0; la salida de una capa es la entrada de la siguiente:

$$\mathbf{a}^{(1)} = \sigma\big(W^{(1)}\mathbf{x} + \mathbf{b}^{(1)}\big) \qquad\qquad \hat y = \sigma\big(W^{(2)}\mathbf{a}^{(1)} + b^{(2)}\big)$$

**Léelo así:** "la capa oculta ($\mathbf{a}^{(1)}$) es la sigmoide de sus pesos por los datos más sus sesgos; la predicción ($\hat y$, *y gorro*) es la sigmoide de los pesos de salida por lo que produjo la capa oculta, más su sesgo". El número entre paréntesis arriba solo dice de qué capa es.

### Ejemplo resuelto: el socio 1
$x_1$ = antigüedad (1.0), $x_2$ = carga de la cuota (0.5), ambas escaladas. **La verdad:** $y = 1$ (sí pagó).

| Capa | Pesos | Sesgos |
|---|---|---|
| Oculta | neurona 1: (0.5, −0.3) · neurona 2: (0.2, 0.8) | 0.1 · −0.1 |
| Salida | (0.7, −0.4) | 0.05 |

In [ ]:
sigmoid = lambda z: 1 / (1 + np.exp(-z))

x  = np.array([1.0, 0.5]); y = 1.0       # member 1 and the truth (paid)
W1 = np.array([[0.5, -0.3],              # hidden neuron 1
               [0.2,  0.8]])             # hidden neuron 2
b1 = np.array([0.1, -0.1])
W2 = np.array([[0.7, -0.4]])             # output neuron
b2 = np.array([0.05])

z1 = W1 @ x + b1;  a1 = sigmoid(z1)       # hidden layer
z2 = W2 @ a1 + b2; y_hat = sigmoid(z2)    # output layer
loss = -np.log(y_hat)                     # cross-entropy (BCE) for y = 1

print("Capa oculta:  z =", z1, " -> a =", a1)
print("Salida:       z =", z2, " -> ŷ =", y_hat)
print("Pérdida:      -log(ŷ) =", loss)

In [ ]:
dibujar_red(W1, b1, W2, b2, x=x, a1=a1, y_hat=y_hat,
            title=f"Forward del socio 1: la red dice {y_hat[0]:.0%} de que paga (y sí pagó)")

**Qué deberías ver:** capa oculta a = (0.6106, 0.6225); predicción **ŷ = 0.5569** (56 % de que paga); pérdida **0.5854 ≈ 0.59**. En el dibujo, el grosor de cada cable es el tamaño del peso (ámbar positivo, azul negativo) y dentro de cada neurona está lo que produce.

### Tarea A · Otro socio, hacia adelante
El **socio 2** tiene antigüedad 0.2 y carga de cuota 1.0, y **no pagó** ($y = 0$). Con la **misma red**:
1. Calcula la capa oculta → `a1_2`.
2. Calcula la predicción → `y_hat_2`.
3. Calcula la pérdida. Ojo: como $y = 0$, la BCE es $-\log(1-\hat y)$ → `loss_2`.

In [ ]:
x_2 = np.array([0.2, 1.0]); y_2 = 0.0
a1_2 = None       # TODO: sigmoid(W1 @ x_2 + b1)
y_hat_2 = None    # TODO: sigmoid(W2 @ a1_2 + b2)
loss_2 = None     # TODO: -log(1 - y_hat_2)

In [ ]:
revisar_ta(a1_2, y_hat_2, loss_2)

---
## Parte B · Backward: la culpa viaja hacia atrás
📽 Láminas 49–51

**Qué vas a hacer:** calcular cuánta "culpa" del error le toca a cada peso.

**La idea:** como en la **cadena del cacao**: el cliente prueba la barra de chocolate y dice "salió amarga" (el error se descubre al final). La culpa se reparte hacia atrás: el tostado recibe culpa según cuánto influyó, y la pasa al fermentado y a la cosecha. Los efectos se multiplican a lo largo de la cadena, como **engranajes**: es la **regla de la cadena**. Tres reglas bastan:

1. **Culpa de la salida:** $\delta^{(2)} = \hat y - y$. *Léelo así:* lo que dijo menos lo que era. Negativa = se quedó corta.
2. **Culpa de una neurona oculta:** la culpa que le llega por su cable (peso × culpa de la salida) **por** la pendiente de su sigmoide, $\sigma'(z) = a(1-a)$.
3. **Gradiente de un peso:** la culpa de su neurona **por** lo que entró por ese cable.

### Ejemplo resuelto: el socio 1

In [ ]:
d2  = y_hat - y                       # rule 1: output blame
dW2 = np.outer(d2, a1); db2 = d2      # rule 3 for the output layer (blame x what came in)

d1  = (W2.T @ d2) * a1 * (1 - a1)     # rule 2: blame sent back through each wire, times sigmoid slope
dW1 = np.outer(d1, x);  db1 = d1      # rule 3 for the hidden layer

print("Culpa de la salida   δ2 =", d2)
print("Gradiente de W2      =", dW2)
print("Culpa capa oculta    δ1 =", d1)
print("Gradiente de W1      =\n", dW1)

**Qué deberías ver:** δ2 = **−0.4431** (la predicción se quedó corta: dijo 0.56 y era 1); gradiente de W2 = (**−0.2706**, **−0.2758**); δ1 = (**−0.0738**, **+0.0417**).
La culpa de la capa oculta es mucho más pequeña: la pendiente de la sigmoide (≈ 0.24) la encoge al pasar hacia atrás. Y la oculta 2 recibe culpa **positiva**: su peso hacia la salida es negativo, así que para que la salida suba, ella debe bajar.

### Tarea B · La culpa de la salida (socio 2)
Con los resultados de tu Tarea A:
1. Culpa de la salida → `d2_2` (regla 1).
2. Gradiente de los pesos de la salida → `dW2_2` (regla 3: `np.outer(d2_2, a1_2)`).

In [ ]:
d2_2 = None     # TODO: rule 1
dW2_2 = None    # TODO: rule 3 for the output layer

In [ ]:
revisar_tb(d2_2, dW2_2)

---
## Parte C · Un paso de aprendizaje y su verificación
📽 Lámina 52

**Qué vas a hacer:** mover los 9 números de la red un pasito cuesta abajo y comprobar que la predicción mejora.

**La idea:** es el descenso de gradiente del viernes (el **Cotopaxi**), con todos los pesos a la vez:

$$w \leftarrow w - \eta \cdot \text{gradiente}$$

**Léelo así:** "a cada peso le restamos η por su culpa". Con η = 0.5, el peso 0.7 de la oculta 1 a la salida queda en $0.7 - 0.5\times(-0.27) = 0.84$.

### Ejemplo resuelto: el socio 1, con η = 0.5

In [ ]:
eta = 0.5
W1n, b1n = W1 - eta * dW1, b1 - eta * db1     # every piece moves against its gradient
W2n, b2n = W2 - eta * dW2, b2 - eta * db2
a1n = sigmoid(W1n @ x + b1n)
y_hat_new = sigmoid(W2n @ a1n + b2n)
print("Peso oculta 1 → salida:", W2[0, 0], "->", round(W2n[0, 0], 4))
print(f"Predicción: {y_hat[0]:.4f} -> {y_hat_new[0]:.4f}")
print(f"Pérdida:    {loss[0]:.4f} -> {-np.log(y_hat_new[0]):.4f}")

In [ ]:
dibujar_red(W1n, b1n, W2n, b2n, x=x, a1=a1n, y_hat=y_hat_new, color_w=MENTA,
            title=f"Después de UN paso (en verde, los 9 números nuevos): {y_hat[0]:.0%} → {y_hat_new[0]:.0%}")

**Qué deberías ver:** el peso 0.7 pasa a **0.8353 ≈ 0.84**; la predicción 0.5569 → **0.6542** (56 % → 65 %) y la pérdida 0.5854 → **0.4243** (0.59 → 0.42).

<details><summary>Para curiosos: ¿cómo sabemos que las culpas están bien? (gradient check)</summary>

Movemos cada peso un poquito ($10^{-5}$) hacia arriba y hacia abajo, medimos cuánto cambia la pérdida y comparamos con nuestras culpas. Si coinciden, las cuentas están bien.

```python
def loss_of(params):
    W1_, b1_, W2_, b2_ = params[:4].reshape(2, 2), params[4:6], params[6:8].reshape(1, 2), params[8:]
    return float(-np.log(sigmoid(W2_ @ sigmoid(W1_ @ x + b1_) + b2_)[0]))
p0 = np.concatenate([W1.ravel(), b1, W2.ravel(), b2])
g_backprop = np.concatenate([dW1.ravel(), db1, dW2.ravel(), db2])
g_numeric = np.array([(loss_of(p0 + 1e-5 * e) - loss_of(p0 - 1e-5 * e)) / 2e-5 for e in np.eye(len(p0))])
print(np.linalg.norm(g_numeric - g_backprop) / np.linalg.norm(g_numeric + g_backprop))   # ~1e-11
```
</details>

### Tarea C · Un paso con el socio 2
1. Calcula la culpa de la capa oculta del socio 2 (regla 2) y su gradiente `dW1_2` (regla 3).
2. Actualiza **las cuatro piezas** (W1, b1, W2, b2) con η = 0.5.
3. Calcula la nueva predicción para el socio 2 → `y_hat_2_nuevo`. ¿Subió o bajó? ¿Tiene sentido?

<details><summary>Pista</summary>

Copia el ejemplo de las Partes B y C y cambia `x → x_2`, `a1 → a1_2`, `d2 → d2_2`. El gradiente de cada sesgo es su culpa (`db = d`).
</details>

In [ ]:
d1_2 = None       # TODO: rule 2
dW1_2 = None      # TODO: rule 3 for the hidden layer
# TODO: update W1, b1, W2, b2 with eta = 0.5 (use new names, e.g. W1_2n)

y_hat_2_nuevo = None    # TODO: forward pass of x_2 with the updated weights

In [ ]:
revisar_tc(y_hat_2_nuevo)

---
## Parte D · TensorFlow hace la cuenta por ti
📽 Lámina 53

**Qué vas a hacer:** pedirle a TensorFlow las mismas culpas de la Parte B, sin escribir ninguna regla.

**La idea:** `tf.GradientTape` funciona como una **grabadora**: graba cada operación de la ida (forward) y después la reproduce al revés aplicando la regla de la cadena. Esto se llama **diferenciación automática** y es como entrena cualquier red de verdad.

### Ejemplo resuelto: los mismos gradientes del socio 1

In [ ]:
# The same weights, now as TensorFlow Variables (numbers TF is allowed to change)
tW1, tb1, tW2, tb2 = [tf.Variable(v) for v in (W1, b1, W2, b2)]

with tf.GradientTape() as tape:                         # 1) press REC: record the forward pass
    t_hidden = tf.sigmoid(tf.linalg.matvec(tW1, tf.constant(x)) + tb1)
    t_yhat = tf.sigmoid(tf.linalg.matvec(tW2, t_hidden) + tb2)
    t_loss = -tf.math.log(t_yhat[0])

gW1, gb1, gW2, gb2 = tape.gradient(t_loss, [tW1, tb1, tW2, tb2])   # 2) play it backwards

print("Gradiente de W1 según TensorFlow:\n", gW1.numpy())
diff = max(np.abs(gW1.numpy() - dW1).max(), np.abs(gW2.numpy() - dW2).max())
print("Diferencia máxima con nuestra cuenta a mano:", diff)

**Qué deberías ver:** la misma matriz de la Parte B y una diferencia del orden de **1e-17**: cero en la práctica (son errores de redondeo).

### Tarea D · Los engranajes en TensorFlow
En las láminas vimos $z = (3x)^2$: por dentro se multiplica por 3 y por fuera, en $x = 1$, por 6. Usa `GradientTape` para obtener $\dfrac{dz}{dx}$ en $x = 1$ → `grad_engranajes`.

<details><summary>Pista</summary>

```python
xv = tf.Variable(1.0)
with tf.GradientTape() as tape:
    zv = (3 * xv) ** 2
grad = tape.gradient(zv, xv)
```
</details>

In [ ]:
grad_engranajes = None    # TODO: use tf.GradientTape

In [ ]:
revisar_td(grad_engranajes)

---
## Parte E · Keras paso a paso: una capa oculta resuelve XOR
📽 Láminas 43, 46, 63

**Qué vas a hacer:** construir tu primera red con Keras, línea por línea, y resolver la **luz de la escalera** (XOR) que el viernes venció al perceptrón.

**La idea:** con Keras armas una red como un **LEGO**: apilas capas, una sobre otra, dentro de `keras.Sequential`. Siempre son tres pasos: **construir → compilar → entrenar**, y después **predecir**.

### E.1 · Los datos
Los datos entran como **arrays de NumPy**: `X` es una tabla (una fila por ejemplo, una columna por dato) e `y` es la respuesta de cada fila.

In [ ]:
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], float)   # 4 examples x 2 switches
y_xor = np.array([0, 1, 1, 0], float)                       # light on (1) only if the switches differ
print("X:", X_xor.shape, "→ 4 ejemplos, 2 datos cada uno | y:", y_xor.shape)

### E.2 · Construir: apilar las capas
- `keras.Sequential([...])` → una red donde la información pasa por las capas **en orden**.
- `keras.Input(shape=(2,))` → la puerta de entrada: cada ejemplo trae **2 números**. No calcula nada.
- `layers.Dense(8, activation="tanh")` → una capa **densa** de **8 neuronas**: cada una conectada con todas las entradas, como la neurona del viernes. `activation` es la función que dobla la línea (`"relu"`, `"tanh"`, `"sigmoid"`…).
- `layers.Dense(1, activation="sigmoid")` → la salida: **una** neurona con sigmoide, que da una **probabilidad** entre 0 y 1.

In [ ]:
keras.utils.set_random_seed(SEED)           # same starting weights for everyone

xor_net = keras.Sequential([
    keras.Input(shape=(2,)),                # 2 inputs: the two switches
    layers.Dense(8, activation="tanh"),     # hidden layer: 8 neurons
    layers.Dense(1, activation="sigmoid"),  # output: probability that the light is on
])
xor_net.summary()                           # a table of the layers and how many numbers each one learns

**Léelo así (la tabla de `summary`):** la capa oculta aprende $2 \times 8 + 8 = 24$ números (un peso por cada cable y un sesgo por neurona) y la salida $8 \times 1 + 1 = 9$. Total: **33 parámetros**, las "perillas" de esta consola de sonido.

### E.3 · Compilar: las reglas del entrenamiento
`compile` no entrena todavía: **configura** cómo se va a entrenar.
- `optimizer` → **cómo** se dan los pasos cuesta abajo. `keras.optimizers.Adam(0.05)` es Adam con `learning_rate` (η) = 0.05.
- `loss` → **qué** se mide para saber qué tan mal vamos: `"binary_crossentropy"` es la BCE del viernes (para sí/no).
- `metrics` → qué más queremos ver mientras entrena (no se usa para aprender): `"accuracy"` = % de aciertos.

In [ ]:
xor_net.compile(optimizer=keras.optimizers.Adam(learning_rate=0.05),
                loss="binary_crossentropy",
                metrics=["accuracy"])

### E.4 · Entrenar: `fit`
`fit` hace **todo lo de las Partes A a C**, miles de veces: adelante → pérdida → atrás → un paso.
- `epochs=500` → 500 vueltas completas por los datos.
- `batch_size=4` → cuántos ejemplos mira antes de dar cada paso (aquí, los 4 a la vez).
- `verbose=0` → que no imprima cada época (usa `1` si quieres verlas).

Devuelve un objeto **History** con la pérdida de cada época.

In [ ]:
hist_xor = xor_net.fit(X_xor, y_xor, epochs=500, batch_size=4, verbose=0)
print(f"Pérdida: época 1 = {hist_xor.history['loss'][0]:.3f}  →  época 500 = {hist_xor.history['loss'][-1]:.3f}")

### E.5 · Predecir: de probabilidad a decisión
`predict` devuelve la **probabilidad** de clase 1 para cada fila. Para decidir, usamos un umbral: **≥ 0.5 → 1**, si no → 0.

In [ ]:
probs = xor_net.predict(X_xor, verbose=0).ravel()     # probabilities, one per row
clases = (probs >= 0.5).astype(int)                   # threshold 0.5 -> decision
for (s1, s2), p, c in zip(X_xor.astype(int), probs, clases):
    print(f"interruptores ({s1}, {s2}) → probabilidad {p:.2f} → luz {'encendida' if c else 'apagada'}")

fig, ax = plt.subplots(1, 2, figsize=(12, 4.3))
ax[0].plot(hist_xor.history["loss"], color=CORAL, lw=2.5)
ax[0].set_xlabel("época"); ax[0].set_ylabel("pérdida (BCE)"); ax[0].set_title("La pérdida baja mientras aprende")
plot_boundary(xor_net, X_xor, y_xor, ax[1], "XOR resuelto: la frontera ya no es una recta")
plt.tight_layout(); plt.show()

**Qué deberías ver:** probabilidades cercanas a **0, 1, 1, 0** → apagada, encendida, encendida, apagada. A la izquierda la pérdida cae; a la derecha, la zona ámbar (encendida) forma una **franja** que deja las dos esquinas apagadas en azul: dos líneas combinadas, como en la lámina 43.

### Tarea E · Cuenta parámetros
Escribe `contar_parametros(capas)`, que recibe una lista como `[2, 16, 16, 1]` (entradas, ocultas…, salida) y devuelve cuántos números aprende la red.

Regla por capa: **entradas × neuronas + neuronas** (un peso por cable y un sesgo por neurona).

<details><summary>Pista</summary>

Recorre parejas consecutivas: `for n_in, n_out in zip(capas[:-1], capas[1:]):`
</details>

In [ ]:
def contar_parametros(capas):
    return None    # TODO

In [ ]:
revisar_te(contar_parametros)

---
## Parte F · Keras completo: la red de las lunas
📽 Láminas 64–67

**Qué vas a hacer:** el flujo completo de un proyecto real: separar datos, construir, compilar, entrenar con validación, mirar las curvas, evaluar y predecir.

### F.1 · Los datos y tres conjuntos
Usamos 600 puntos con forma de **dos lunas** (clase 1 ámbar, clase 0 azul). Los separamos como cuando estudias:
- **entrenamiento** → los ejercicios con los que estudias (la red aprende de ellos);
- **validación** → los **simulacros** que haces mientras estudias, para ver si de verdad entendiste o solo memorizaste;
- **prueba** → el **examen final**: datos que la red nunca vio, se usan una sola vez al final.

In [ ]:
X_m, y_m = make_moons(600, noise=0.2, random_state=SEED)       # 600 points, 2 features
X_train, X_test, y_train, y_test = train_test_split(X_m, y_m, test_size=0.2, random_state=SEED, stratify=y_m)
print("entrenamiento:", X_train.shape, "| prueba (examen final):", X_test.shape)

plt.figure(figsize=(5.5, 4.2))
plt.scatter(*X_train.T, c=np.where(y_train == 1, AMBAR, AZUL), edgecolor=TINTA, s=25, lw=.4)
plt.title("Las lunas: ninguna recta las separa"); plt.xlabel("dato 1"); plt.ylabel("dato 2"); plt.show()

### F.2 · Construir y contar
Dos capas ocultas de 16 neuronas con **ReLU** (la válvula: la culpa llega entera hacia atrás) y una salida con sigmoide.

**Léelo así (los parámetros):** capa 1: $2\times16+16 = 48$ · capa 2: $16\times16+16 = 272$ · salida: $16\times1+1 = 17$ · **total 337**.

In [ ]:
keras.utils.set_random_seed(SEED)
model = keras.Sequential([
    keras.Input(shape=(2,)),                 # 2 input features
    layers.Dense(16, activation="relu"),     # hidden layer 1: 2*16 + 16 = 48 parameters
    layers.Dense(16, activation="relu"),     # hidden layer 2: 16*16 + 16 = 272
    layers.Dense(1, activation="sigmoid"),   # output: 16*1 + 1 = 17 -> probability of class 1
])
model.summary()
print("Total de parámetros:", model.count_params())

### F.3 · Compilar
Adam (el optimizador de la carrera, lámina 58) con `learning_rate=0.01`, la BCE como pérdida y la exactitud como métrica.

In [ ]:
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.01),
              loss="binary_crossentropy",
              metrics=["accuracy"])

### F.4 · Entrenar con simulacros
- `epochs=60` → 60 vueltas por los datos de entrenamiento.
- `batch_size=32` → da un paso cada 32 ejemplos (una **encuesta**, no un censo: lámina 57).
- `validation_split=0.2` → separa el 20 % del entrenamiento como **simulacros**: la red no aprende de ellos, solo se mide.

In [ ]:
history = model.fit(X_train, y_train, epochs=60, batch_size=32, validation_split=0.2, verbose=0)

h = history.history                      # a dictionary: one list per measure, one value per epoch
print("Lo que guardó History:", list(h.keys()))
print(f"Época 60 → exactitud entrenamiento {h['accuracy'][-1]:.1%} | validación (simulacros) {h['val_accuracy'][-1]:.1%}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.9))
ax[0].plot(h["loss"], color=AMBAR, lw=2.5, label="entrenamiento")
ax[0].plot(h["val_loss"], color=LILA, lw=2.5, label="validación (simulacros)")
ax[0].set_xlabel("época"); ax[0].set_ylabel("pérdida"); ax[0].set_title("La pérdida baja época a época"); ax[0].legend()
ax[1].plot(h["accuracy"], color=AMBAR, lw=2.5, label="entrenamiento")
ax[1].plot(h["val_accuracy"], color=LILA, lw=2.5, label="validación (simulacros)")
ax[1].set_xlabel("época"); ax[1].set_ylabel("exactitud"); ax[1].set_ylim(.5, 1); ax[1].set_title("La exactitud sube"); ax[1].legend()
plt.tight_layout(); plt.show()

**Qué deberías ver:** **337 parámetros**; exactitud de validación cercana a **95 %**; y dos curvas que bajan (pérdida) y suben (exactitud) **juntas**. Si la de entrenamiento siguiera mejorando y la de validación empeorara, la red estaría **memorizando** en vez de aprender (sobreajuste: semana 3).

### F.5 · El examen final y predecir
- `model.evaluate(X, y)` → pérdida y exactitud en datos que la red **nunca vio**.
- `model.predict(X)` → la **probabilidad** de clase 1 para cada punto; con el umbral 0.5 la convertimos en decisión.

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Examen final (prueba): pérdida {test_loss:.3f} | exactitud {test_acc:.1%}")

probs_test = model.predict(X_test[:5], verbose=0).ravel()
print("Primeros 5 del examen:")
for p, c, real in zip(probs_test, (probs_test >= 0.5).astype(int), y_test[:5]):
    print(f"  probabilidad {p:.2f} → predice {c} | era {real} {'✔' if c == real else '✘'}")

fig, ax = plt.subplots(figsize=(6, 4.8))
plot_boundary(model, X_m, y_m, ax, "Lunas: la frontera se curva"); plt.show()

**Qué deberías ver:** una exactitud de prueba parecida a la de validación (≈ 95 %), las primeras predicciones casi todas con ✔ y una frontera negra curva que sigue la forma de las lunas.

---
### 🏁 Mini-reto 2 · Los círculos
📽 Lámina 68

Puntos de una clase **dentro** de un círculo y de la otra **alrededor**. Ninguna recta los separa.

**Tu misión:** construye, compila y entrena una red con Keras que llegue a **≥ 90 % de exactitud en validación**. Usa la Parte F como plantilla.

**Para pensar:**
1. Primero prueba una red **sin capas ocultas** (solo `layers.Dense(1, activation="sigmoid")`). ¿Qué exactitud logra? ¿Por qué?
2. Luego agrega capas ocultas. ¿Cuántas neuronas necesitas como mínimo?

In [ ]:
X_c, y_c = make_circles(600, noise=0.1, factor=0.5, random_state=SEED)
Xc_tr, Xc_va, yc_tr, yc_va = train_test_split(X_c, y_c, test_size=0.25, random_state=SEED, stratify=y_c)
plt.figure(figsize=(4.8, 4.5)); plt.scatter(*X_c.T, c=np.where(y_c == 1, AMBAR, AZUL), edgecolor=TINTA, s=25, lw=.4)
plt.title("Los círculos: ¿cómo los separas?"); plt.show()

In [ ]:
keras.utils.set_random_seed(SEED)
modelo_circulos = None
# TODO 1) BUILD:   modelo_circulos = keras.Sequential([keras.Input(shape=(2,)), ..., layers.Dense(1, activation="sigmoid")])
# TODO 2) COMPILE: modelo_circulos.compile(optimizer=keras.optimizers.Adam(0.01), loss="binary_crossentropy", metrics=["accuracy"])
# TODO 3) TRAIN:   hist_c = modelo_circulos.fit(Xc_tr, yc_tr, validation_data=(Xc_va, yc_va), epochs=60, batch_size=32, verbose=0)

In [ ]:
revisar_mini_reto_2(modelo_circulos, Xc_va, yc_va)
if modelo_circulos is not None:
    fig, ax = plt.subplots(figsize=(5, 4.6)); plot_boundary(modelo_circulos, X_c, y_c, ax, "Tu frontera"); plt.show()

---
## Parte G · ¿Qué optimizador llega primero?
📽 Lámina 58

**Qué vas a hacer:** entrenar la misma red, con los mismos pesos iniciales, usando tres optimizadores.

**La idea:** el optimizador es la **forma de caminar** cuesta abajo:
- **SGD:** el caminante. Pasos fijos en la dirección de la pendiente.
- **Momentum:** una bola que rueda (o la bicicleta en bajada): acumula impulso y no se frena en cada piedrita.
- **Adam:** zapatos a la medida de cada terreno: ajusta el paso de **cada peso** por separado.

### Ejemplo resuelto

In [ ]:
def make_model():
    keras.utils.set_random_seed(SEED)   # same starting weights for a fair comparison
    return keras.Sequential([keras.Input(shape=(2,)), layers.Dense(16, activation="relu"),
                             layers.Dense(16, activation="relu"), layers.Dense(1, activation="sigmoid")])

optimizers = {"SGD": keras.optimizers.SGD(0.05),
              "Momentum": keras.optimizers.SGD(0.05, momentum=0.9),
              "Adam": keras.optimizers.Adam(0.01)}
colores = {"SGD": "gray", "Momentum": TINTA, "Adam": LILA}
resultados_g = {}
plt.figure(figsize=(7.5, 4))
for name, opt in optimizers.items():
    m = make_model(); m.compile(optimizer=opt, loss="binary_crossentropy")
    hh = m.fit(X_train, y_train, validation_split=0.2, epochs=40, batch_size=32, verbose=0)
    resultados_g[name] = round(hh.history["val_loss"][-1], 3)
    plt.plot(hh.history["val_loss"], lw=2.5, color=colores[name], label=name)
plt.xlabel("época"); plt.ylabel("pérdida de validación"); plt.legend(); plt.title("Misma red, tres formas de caminar"); plt.show()
print(resultados_g)

**Qué deberías ver:** SGD baja despacio; Momentum y Adam llegan mucho antes a una pérdida baja. Ojo, como en la lectura de tendencia: con otros `learning_rate` el orden puede cambiar. **Comparar sin ajustar a todos con el mismo cuidado es una carrera injusta.**

### Tarea G · Lee las curvas
¿Qué optimizador terminó con la pérdida de validación **más alta**? Escribe `"SGD"`, `"Momentum"` o `"Adam"` en `peor_optimizador`.

In [ ]:
peor_optimizador = None    # TODO

In [ ]:
revisar_tg(peor_optimizador, resultados_g)

---
## Resumen en 5 líneas
1. Una **capa** es `W @ x + b` y una activación; una **red** son capas en fila (forward).
2. La **pérdida** dice qué tan mal vamos; la **culpa** viaja hacia atrás con la regla de la cadena (backward) y cada peso da un paso en contra de su gradiente.
3. **TensorFlow** es el motor: calcula esas derivadas solo con `GradientTape`. **Keras** es el volante: construir → compilar → entrenar → predecir.
4. Las capas ocultas con activación permiten **fronteras curvas**: XOR, lunas y círculos.
5. Se estudia con **entrenamiento**, se mide con **validación** (simulacros) y se juzga una vez con **prueba** (el examen final).

---
## 🏁 Tareas finales

### Tarea final 1 · Usa tu modelo con socios nuevos
`NUEVOS` (ya cargado) tiene 3 puntos que la red de las lunas nunca vio.
1. Calcula sus probabilidades con `model.predict(NUEVOS, verbose=0).ravel()` → `probs_nuevos`.
2. Conviértelas en decisiones con el umbral 0.5 (1 o 0) → `clases_nuevas`.

In [ ]:
print(NUEVOS)
probs_nuevos = None     # TODO
clases_nuevas = None    # TODO

In [ ]:
revisar_tf1(model, probs_nuevos, clases_nuevas)

### Tarea final 2 · Experimenta (sin revisión automática)
Copia la Parte F en una celda nueva y cambia **una cosa a la vez**. Anota la exactitud de validación de cada intento:
1. `Dense(16)` → `Dense(4)` en las dos capas ocultas. ¿Cuántos parámetros quedan? ¿Baja la exactitud?
2. `activation="relu"` → `"tanh"`.
3. `learning_rate=0.01` → `1.0`. ¿Qué pasa con la curva de pérdida? (Piensa en **la ducha**: abrir de golpe el agua caliente.)
4. `make_moons(600, noise=0.2)` → `noise=0.35`. ¿Qué le pasa a la frontera?

Y si todavía no lo hiciste, vuelve al **Mini-reto 2** (los círculos) de la Parte F.

---
**¿Te quedó alguna duda?** Repasa las slides interactivas y la guía de estudio desde la página del curso, y juega el **quiz final**.